# aula01 — teoria resolvida
Código executável em R. As evidências da execução por Rscript estão em estrutura/resultados.


In [ ]:
if (!file.exists("estrutura/codigos")) setwd("../..") # Posiciona na raiz do repositorio.
if (.Platform$OS.type == "windows") invisible(Sys.setlocale("LC_CTYPE", "English_United States.utf8")) # Habilita UTF-8.
source("estrutura/codigos/motor-professor/motor01.R", encoding = "UTF-8") # Carrega as aulas anteriores.
docs <- docs_aula(); cfg <- cfg_aula(); ix <- montar(docs, cfg) # Recria o exemplo canonico.


In [ ]:
docs <- c(                                                          # vetor de caracteres, com nomes
  d1 = "recuperacao de informacao ordena documentos por relevancia", # 7 palavras
  d2 = "o modelo de espaco vetorial representa documentos como vetores", # 9
  d3 = "bm25 e um modelo probabilistico de ranqueamento de texto",   # 9
  d4 = "aprendizado estatistico fundamenta a recuperacao moderna",   # 6
  d5 = "o indice invertido acelera a busca em muitos documentos",    # 9
  d6 = "embeddings capturam a semantica de palavras e documentos",   # 8
  d7 = "a avaliacao mede a relevancia dos resultados da busca",      # 9
  d8 = "ciencia de dados combina estatistica e programacao"          # 7
)                                                                   # fecha o c(...)


In [ ]:
tokenizar <- function(texto) {                    # (já carregada pelo motor01.R)
  t <- unlist(strsplit(tolower(texto), "\\s+"))   # minúsculas; quebra em um ou mais espaços
  t[nzchar(t)]                                    # tira o "" que sobra de espaço na ponta
}


In [ ]:
tokens <- lapply(docs, tokenizar)   # aplica a cada documento -> lista de 8 vetores
tokens[["d1"]]                      # os termos de d1


In [ ]:
vocab <- sort(unique(unlist(tokens)))   # junta tudo, tira repetidos, ordena
length(vocab)                           # quantos termos distintos


In [ ]:
freq <- table(unlist(tokens))          # conta quantas vezes cada termo aparece no corpus todo
sort(freq, decreasing = TRUE)[1:6]     # do mais frequente ao menos; só os 6 primeiros


In [ ]:
docs9 <- c(docs, d9 = "porto de santos")                     # uma cópia com um 9º documento
length(sort(unique(unlist(lapply(docs9, tokenizar)))))       # o vocabulário cresce quanto?


In [ ]:
tdm <- sapply(tokens, function(tk) {              # para cada documento (tk = seus tokens)...
  as.integer(table(factor(tk, levels = vocab)))   # ...conta cada termo do vocab neste doc
})                                                # sapply empilha os 8 vetores em colunas
rownames(tdm) <- vocab                            # dá nome às linhas: os termos


In [ ]:
busca_booleana <- function(termo, tdm) {                # recebe um termo e a matriz
  termo <- tolower(termo)                               # mesma normalização da tokenização
  if (!termo %in% rownames(tdm)) return(character(0))   # termo inexistente: devolve vazio
  colnames(tdm)[tdm[termo, ] > 0]                       # nomes das colunas com contagem > 0
}                                                       # fim da função


In [ ]:
df <- 1:8                                    # em quantos docs o termo aparece: 1 a 8
plot(df, log(8 / df), type = "b",            # o idf de cada caso, pontos ligados por linha
     xlab = "df", ylab = "idf = log(8/df)")  # nomes dos eixos


In [ ]:
N   <- ncol(tdm)             # 8 documentos
df  <- rowSums(tdm > 0)      # em quantos docs cada termo aparece (TRUE conta como 1)
idf <- log(N / df)           # um idf por termo: vetor de 45
tfidf <- tdm * idf           # cada linha da matriz vezes o seu idf


In [ ]:
grep("recupera", docs)             # acha "recuperacao": casa PEDACO -> [1] 1 4
busca_booleana("recupera", tdm)    # o TERMO "recupera" nao existe -> character(0)
